# 3.3 — Разбор кейсов и интерпретация

**Папка 3, подноутбук 3.** Качественный анализ: репрезентативные траектории, полосы
неопределённости DPI-Flow, активация события EVT-NeuralSSM, апостериорные распределения
выведенных параметров и карта скрытых состояний. Все рисунки — на английском с единицами
измерения.

## Окружение, данные и модели

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Найти корень репозитория по наличию pyproject.toml вверх по дереву."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display

from liquefaction_ai.viz import register_theme

register_theme()

# Если True — все фигуры сохраняются в results/figs (.html и .png)
SAVE_FIGS = True
DATA_DIR = REPO_ROOT / "data" / "demo_run"
MODELS_DIR = REPO_ROOT / "models"

import torch

from liquefaction_ai import load_population_artifact, prepare_benchmark_dataset
from liquefaction_ai.training import load_model_metadata, load_weights_into
from liquefaction_ai.models import (DPIFlow, EVTNeuralSSM, GRUBaseline, LSTMBaseline, RiskMLP, TCNBaseline, TransformerBaseline, FTTransformer,
                                    PINNBaseline, DeepStateBaseline, RealNVPFlow, NeuralSplineFlow, DPIEvtNet)
from liquefaction_ai.evaluation import collect_outputs, compute_metrics, english_metric_table

CLASS_REGISTRY = {"RiskMLP": RiskMLP, "GRUBaseline": GRUBaseline, "TCNBaseline": TCNBaseline, "LSTMBaseline": LSTMBaseline, "TransformerBaseline": TransformerBaseline, "FTTransformer": FTTransformer, "PINNBaseline": PINNBaseline, "DeepStateBaseline": DeepStateBaseline, "RealNVPFlow": RealNVPFlow, "NeuralSplineFlow": NeuralSplineFlow,
                  "DPIFlow": DPIFlow, "EVTNeuralSSM": EVTNeuralSSM, "DPIEvtNet": DPIEvtNet}
MODEL_NAMES = ["mlp_risk", "gru", "tcn", "lstm", "transformer", "ft_transformer", "pinn", "deepstate", "realnvp", "nsf", "dpi_flow", "evt_ssm", "dpi_evt"]

population, config = load_population_artifact(DATA_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
benchmark = prepare_benchmark_dataset(population, config, device)
test = benchmark["test"]


def load_trained(name):
    """Восстановить модель по сохранённым гиперпараметрам и весам."""
    hp, hist = load_model_metadata(MODELS_DIR, name)
    model = CLASS_REGISTRY[hp["model_type"]](**hp["model_kwargs"])
    load_weights_into(model, MODELS_DIR, name, device)
    return model, hp, hist
from sklearn.decomposition import PCA
from liquefaction_ai.constants import LOAD_DISPLAY_NAMES_EN, SOIL_DISPLAY_NAMES_EN
from liquefaction_ai.data import iterate_minibatches
from liquefaction_ai.viz import histogram_grid, line_with_bands, lines, scatter

dpi, _, _ = load_trained("dpi_flow"); evt, _, _ = load_trained("evt_ssm"); tcn, _, _ = load_trained("tcn")
dpi_out = collect_outputs(dpi, test, config, device)
evt_out = collect_outputs(evt, test, config, device)
tcn_out = collect_outputs(tcn, test, config, device)
# Эталон траектории — измеренное PPR (как в реальном опыте); истинный триггер опционален
cycles = test["cycles"].cpu().numpy(); r_true = test["r_obs"].cpu().numpy()
g_true = test["g_true"].cpu().numpy() if "g_true" in test else None
tm = test["meta"].reset_index(drop=True)
# На реальных данных нет латентного риска — используем наблюдаемый прокси (пик PPR)

## Representative trajectories: ground truth vs models

In [2]:
pick = (tm[tm["liq_label"] == 1].sort_values("PPR_max_true", ascending=False).head(2).index.tolist()
        + tm[tm["liq_label"] == 0].sort_values("PPR_max_true").head(1).index.tolist())
for idx in pick:
    soil = SOIL_DISPLAY_NAMES_EN.get(tm.loc[idx, "soil_type"], tm.loc[idx, "soil_type"])
    load = LOAD_DISPLAY_NAMES_EN.get(tm.loc[idx, "load_mode"], tm.loc[idx, "load_mode"])
    series = [
        {"x": cycles[idx], "y": r_true[idx], "name": "measured", "color": "#1f2937", "width": 3},
        {"x": cycles[idx], "y": tcn_out["traj_mean"][idx], "name": "TCN", "color": "#fd7e14"},
        {"x": cycles[idx], "y": dpi_out["traj_mean"][idx], "name": "DPI-Flow", "color": "#0b6efd"},
        {"x": cycles[idx], "y": evt_out["traj_mean"][idx], "name": "EVT-NeuralSSM", "color": "#6610f2"},
    ]
    lines(series, title=f"PPR trajectory — {soil} | {load} (case {idx})",
          xlabel="Number of cycles, N", ylabel="Pore-pressure ratio, PPR (–)", logx=True, hline=1.0,
          save=SAVE_FIGS, fig_id=f"3_3_trajectory_case_{idx}").show()

[save_figure] PNG для '3_3_trajectory_case_48' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[save_figure] PNG для '3_3_trajectory_case_69' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[save_figure] PNG для '3_3_trajectory_case_61' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## DPI-Flow uncertainty bands

In [3]:
std = np.sqrt(np.exp(dpi_out["traj_logvar"]))
for idx in pick[:2]:
    mean = dpi_out["traj_mean"][idx]
    lower = np.clip(mean - 1.64 * std[idx], 0, 1.05); upper = np.clip(mean + 1.64 * std[idx], 0, 1.05)
    line_with_bands(cycles[idx], mean, lower, upper,
                    extra=[{"x": cycles[idx], "y": r_true[idx], "name": "measured", "color": "#1f2937", "width": 2.6}],
                    mean_name="DPI-Flow mean", band_name="90% interval",
                    title=f"DPI-Flow 90% uncertainty interval (case {idx})",
                    xlabel="Number of cycles, N", ylabel="Pore-pressure ratio, PPR (–)", logx=True,
                    save=SAVE_FIGS, fig_id=f"3_3_uncertainty_case_{idx}").show()

[save_figure] PNG для '3_3_uncertainty_case_48' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[save_figure] PNG для '3_3_uncertainty_case_69' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## EVT-NeuralSSM event activation

In [4]:
idx = pick[0]
series = []
if g_true is not None:
    series.append({"x": cycles[idx], "y": g_true[idx], "name": "true trigger", "color": "#d63384", "width": 2.6})
series += [
    {"x": cycles[idx], "y": evt_out["g"][idx], "name": "predicted trigger (latent)", "color": "#6610f2", "dash": "dash"},
    {"x": cycles[idx], "y": evt_out["traj_mean"][idx], "name": "predicted PPR", "color": "#198754", "width": 1.8},
]
lines(series, title=f"EVT-NeuralSSM: damage-to-event transition (case {idx})",
      xlabel="Number of cycles, N", ylabel="Activation / PPR (–)", logx=True,
      save=SAVE_FIGS, fig_id="3_3_evt_activation").show()

[save_figure] PNG для '3_3_evt_activation' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## DPI-Flow parameter posteriors

In [5]:
dpi.eval()
batch = next(iterate_minibatches(test, batch_size=test["static"].shape[0], device=device, shuffle=False))
with torch.no_grad():
    theta = dpi.forward_batch(batch)["theta_params"]
theta_df = pd.DataFrame({
    "Hyperbolic family weight (–)": theta["weights"][:, 0].cpu().numpy(),
    "Logarithmic family weight (–)": theta["weights"][:, 3].cpu().numpy(),
    "Damage rate λ (1/cycle)": theta["lambda_damage"].cpu().numpy(),
    "PPR growth rate α (1/cycle)": theta["alpha"].cpu().numpy(),
})
histogram_grid(theta_df, columns=list(theta_df.columns), n_cols=2,
               title="DPI-Flow inferred parameter posteriors",
               save=SAVE_FIGS, fig_id="3_3_dpi_parameter_posteriors").show()

[save_figure] PNG для '3_3_dpi_parameter_posteriors' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## EVT-NeuralSSM latent-state map

In [6]:
state = np.column_stack([evt_out["z"][:, -1], evt_out["traj_mean"][:, -1],
                         evt_out["c"][:, -1], evt_out["g"][:, -1]])
proj = PCA(n_components=2, random_state=config.seed).fit_transform(state)
scatter(proj[:, 0], proj[:, 1], color=tm["PPR_max_true"], color_label="Peak PPR (–)",
        title="EVT-NeuralSSM latent-state map (PCA)", xlabel="PC1", ylabel="PC2",
        save=SAVE_FIGS, fig_id="3_3_latent_state_map").show()

[save_figure] PNG для '3_3_latent_state_map' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Итог

Структурированные модели точнее и интерпретируемее: калиброванная неопределённость, явный
момент события и осмысленные физические параметры. Это завершает пайплайн анализа.